# Developing Classification Methods for Target Role and Current Role
Classifying the "target role" and "current role" we parsed from relevant post titles.

In [ ]:
import pandas as pd

#### Parsing Current Role & Target Role

In [ ]:
parsed_title_parts["current_role"].value_counts().head(5)

current_role
Unemployed           1816
Recent Graduate       368
Student               363
Software Engineer     262
unemployed            147
Name: count, dtype: int64

In [ ]:
parsed_title_parts["target_role"].value_counts().head(5)

target_role
Software Engineer           343
Software Developer           80
Data Analyst                 80
Senior Software Engineer     55
Data Scientist               54
Name: count, dtype: int64

In [ ]:
# combines all target and current roles
all_roles = pd.concat([parsed_title_parts["current_role"], parsed_title_parts["target_role"]])
all_unique_roles = list(set(all_roles.unique()))
all_roles.value_counts().head(5)

Unemployed           1853
Software Engineer     605
Recent Graduate       374
Student               371
unemployed            157
Name: count, dtype: int64

##### Defining Taxonomy for Standardizing Job Roles
Using the SOC established taxonomy along with 6 amended groups for unrepresented types found in Reddit data.

In [ ]:
# loads official soc taxonomy
soc_taxonomy = pd.read_excel("data/sample/soc_structure_2018.xlsx", skiprows=7)

In [ ]:
# cleans up column names
soc_taxonomy.columns = [
    "Major Group",
    "Minor Group",
    "Broad Group",
    "Detailed Occupation",
    "Name"
]

In [ ]:
# investigating counts before amending
soc_taxonomy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1453 entries, 0 to 1452
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Major Group          24 non-null     str   
 1   Minor Group          98 non-null     object
 2   Broad Group          459 non-null    object
 3   Detailed Occupation  872 non-null    str   
 4   Name                 1453 non-null   str   
dtypes: object(2), str(3)
memory usage: 113.3+ KB


In [ ]:
# adds specific states to SOC which don't exist yet
import numpy as np

extra_rows = pd.DataFrame(
    [
        # Major Group Header (mirrors official BLS major groups like 11-0000, 15-0000)
        {
            "Major Group": "99-0000",
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": np.nan,
            "Name": "Non-Occupational and Transition Statuses",
        },
        # Detailed Occupation Rows
        {
            "Major Group": np.nan,
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": "99-1001",
            "Name": (
                "Unemployed, Laid Off, Looking for Work, Seeking Employment"
            ),
        },
        {
            "Major Group": np.nan,
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": "99-1002",
            "Name": "Student, Intern, College Student, Recent Graduate",
        },
        {
            "Major Group": np.nan,
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": "99-1003",
            "Name": "Self Employed, Freelancer, Founder, Business Owner",
        },
        {
            "Major Group": np.nan,
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": "99-1004",
            "Name": "Anything, Undecided, General, Whatever, Any Job",
        },
        {
            "Major Group": np.nan,
            "Minor Group": np.nan,
            "Broad Group": np.nan,
            "Detailed Occupation": "99-1005",
            "Name": "Volunteer, Caregiver, Homemaker, Retired",
        },
    ]
)

soc_taxonomy = pd.concat([soc_taxonomy, extra_rows], ignore_index=True)

In [ ]:
# investigating numbers after amending
soc_taxonomy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Major Group          25 non-null     str   
 1   Minor Group          98 non-null     object
 2   Broad Group          459 non-null    object
 3   Detailed Occupation  877 non-null    str   
 4   Name                 1459 non-null   str   
dtypes: object(2), str(3)
memory usage: 113.8+ KB


##### Classifying Sample Data based on Taxonomy

In [ ]:
soc_taxonomy

,Major Group,Minor Group,Broad Group,Detailed Occupation,Name
0,11-0000,NaN,NaN,NaN,Management Occupations
1,NaN,11-1000,NaN,NaN,Top Executives
2,NaN,NaN,11-1010,NaN,Chief Executives
3,NaN,NaN,NaN,11-1011,Chief Executives
4,NaN,NaN,11-1020,NaN,General and Operations Managers
...,...,...,...,...,...
1454,NaN,NaN,NaN,99-1001,"Unemployed, Laid Off, Looking for Work, Seekin..."
1455,NaN,NaN,NaN,99-1002,"Student, Intern, College Student, Recent Graduate"
1456,NaN,NaN,NaN,99-1003,"Self Employed, Freelancer, Founder, Business O..."
1457,NaN,NaN,NaN,99-1004,"Anything, Undecided, General, Whatever, Any Job"


In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Define and load the instruction model you downloaded earlier:
model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-1.5B-instruct")

/Users/saakethkesireddy/Desktop/job_journal/resume_dataset/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1123 17:37:12.512000 8566 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5670.27it/s]
[transformers] Qwen2Model LOAD REPORT from: Alibaba-NLP/gte-Qwen2-1.5B-instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------------------------
# 1. PREPARE ANCHORS FROM YOUR TABLE
# ------------------------------------------------------------------------------
df_soc = soc_taxonomy.copy()

# Forward-fill Major Group so each detailed role knows its parent family
df_soc["Major Group"] = df_soc["Major Group"].ffill()

# Filter to rows that have a specific job code (Detailed Occupation or Broad Group)
# This includes all official jobs + your 99-1001 through 99-1005 statuses
anchor_df = df_soc[
    df_soc["Detailed Occupation"].notna() | df_soc["Broad Group"].notna()
].copy()
anchor_df["soc_code"] = anchor_df["Detailed Occupation"].combine_first(
    anchor_df["Broad Group"]
)
anchor_df = anchor_df.drop_duplicates(subset=["Name"]).reset_index(drop=True)

print(
    f"Total anchor categories: {len(anchor_df)} (including 99-1001 to 99-1005)"
)

# ------------------------------------------------------------------------------
# 2. LOAD MODEL & EMBED ANCHORS
# ------------------------------------------------------------------------------
# (Skip this line if you already defined model earlier)
model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-1.5B-instruct")

print("Embedding anchor titles...")
anchor_embeddings = model.encode(
    anchor_df["Name"].tolist(), normalize_embeddings=True, show_progress_bar=True
)

# ------------------------------------------------------------------------------
# 3. EMBED YOUR RESUME ROLES
# ------------------------------------------------------------------------------
print(f"Embedding {len(all_unique_roles)} unique resume roles...")
role_embeddings = model.encode(
    all_unique_roles,
    prompt="Represent this career title for clustering by occupation and industry: ",
    normalize_embeddings=True,
    show_progress_bar=True,
)

# ------------------------------------------------------------------------------
# 4. COMPUTE COSINE SIMILARITY & CLASSIFY
# ------------------------------------------------------------------------------
# Matrix multiplication (cosine similarity of normalized vectors)
sim_matrix = role_embeddings @ anchor_embeddings.T

# Top 1 Match
best_match_idx = sim_matrix.argmax(axis=1)
confidence_scores = sim_matrix.max(axis=1)

# Runner-up (2nd best match) to detect ambiguity
sorted_indices = np.argsort(sim_matrix, axis=1)
runner_up_idx = sorted_indices[:, -2]
runner_up_scores = np.take_along_axis(
    sim_matrix, sorted_indices[:, -2:], axis=1
)[:, 0]

# ------------------------------------------------------------------------------
# 5. BUILD THE FINAL CLASSIFIED DATAFRAME WITH DECISION POLICIES
# ------------------------------------------------------------------------------
matched_top1 = anchor_df.iloc[best_match_idx].reset_index(drop=True)
matched_top2 = anchor_df.iloc[runner_up_idx].reset_index(drop=True)

results = []
for i in range(len(all_unique_roles)):
    score = float(confidence_scores[i])
    margin = score - float(runner_up_scores[i])

    # System Decision Policy:
    if score < 0.40:
        decision_status = "REJECTED_LOW_CONFIDENCE"
        category = "Other / Unclassified"
        code = "99-9999"
    elif margin < 0.04 and score < 0.65:
        decision_status = "FLAGGED_AMBIGUOUS"
        category = f"{matched_top1.iloc[i]['Name']} / {matched_top2.iloc[i]['Name']}"
        code = matched_top1.iloc[i]["soc_code"]
    elif score >= 0.50:
        decision_status = "AUTO_ACCEPTED_HIGH"
        category = matched_top1.iloc[i]["Name"]
        code = matched_top1.iloc[i]["soc_code"]
    else:
        decision_status = "AUTO_ACCEPTED_MEDIUM"
        category = matched_top1.iloc[i]["Name"]
        code = matched_top1.iloc[i]["soc_code"]

    results.append(
        {
            "raw_role": all_unique_roles[i],
            "category": category,
            "soc_code": code,
            "major_group": matched_top1.iloc[i]["Major Group"],
            "confidence": round(score, 3),
            "decision_status": decision_status,
        }
    )

df_classified = pd.DataFrame(results)

# ------------------------------------------------------------------------------
# 6. VIEW RESULTS
# ------------------------------------------------------------------------------
print("\n" + "=" * 30 + " TOP 10 CLASSIFIED CATEGORIES " + "=" * 30)
print(df_classified["category"].value_counts().head(10))

print("\n" + "=" * 30 + " SAMPLE PREDICTIONS " + "=" * 30)
print(
    df_classified.sample(15)[
        ["raw_role", "category", "soc_code", "confidence", "decision_status"]
    ].to_string(index=False)
)

Total anchor categories: 1081 (including 99-1001 to 99-1005)


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5565.99it/s]
[transformers] Qwen2Model LOAD REPORT from: Alibaba-NLP/gte-Qwen2-1.5B-instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding anchor titles...


Batches: 100%|██████████| 34/34 [00:25<00:00,  1.34it/s]


Embedding 7786 unique resume roles...


Batches: 100%|██████████| 244/244 [04:55<00:00,  1.21s/it]



============================== TOP 10 CLASSIFIED CATEGORIES ==============================
category
Software and Web Developers, Programmers, and Testers                                                     503
Student, Intern, College Student, Recent Graduate                                                         127
Other / Unclassified                                                                                      119
Transportation, Storage, and Distribution Managers / Human Resources Managers                             102
Human Resources Managers / Transportation, Storage, and Distribution Managers                              93
Market Research Analysts and Marketing Specialists                                                         84
Software and Web Developers, Programmers, and Testers / Airline Pilots, Copilots, and Flight Engineers     79
Miscellaneous Computer Occupations                                                                         55
Bookkeeping, Accoun

In [ ]:
df_classified["soc_code"].value_counts()

soc_code
15-1250    971
15-1290    351
99-1002    269
11-3070    260
11-3120    209
          ... 
55-3019      1
51-5111      1
27-2041      1
55-2010      1
25-2011      1
Name: count, Length: 346, dtype: int64

In [ ]:
df_classified["major_group"].value_counts()

major_group
15-0000    1565
13-0000     952
43-0000     659
17-0000     619
99-0000     616
11-0000     599
25-0000     520
41-0000     439
27-0000     294
19-0000     247
51-0000     206
53-0000     178
21-0000     155
29-0000     129
31-0000     118
49-0000     116
55-0000      87
39-0000      76
35-0000      59
47-0000      50
23-0000      50
33-0000      25
45-0000      16
37-0000      11
Name: count, dtype: int64

In [ ]:
df_classified[df_classified["major_group"] == "99-0000"]["category"].value_counts()

category
Student, Intern, College Student, Recent Graduate                                                                                                          127
Unemployed, Laid Off, Looking for Work, Seeking Employment / Miscellaneous Computer Occupations                                                             49
Unemployed, Laid Off, Looking for Work, Seeking Employment                                                                                                  48
Anything, Undecided, General, Whatever, Any Job                                                                                                             41
Self Employed, Freelancer, Founder, Business Owner                                                                                                          33
                                                                                                                                                          ... 
Student, Intern, College Student, Rec

In [ ]:
df_classified[df_classified["category"] == "Unemployed, Laid Off, Looking for Work, Seeking Employment"]

,raw_role,category,soc_code,major_group,confidence,decision_status
142,Sales Executive/Unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.606,AUTO_ACCEPTED_HIGH
160,Currently Uneployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.632,AUTO_ACCEPTED_HIGH
167,Former TV Producer/Currently Unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.605,AUTO_ACCEPTED_HIGH
190,Recently Unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.647,AUTO_ACCEPTED_HIGH
260,About to Be Unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.648,AUTO_ACCEPTED_HIGH
504,currently unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.580,AUTO_ACCEPTED_HIGH
1305,unemplyed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.495,AUTO_ACCEPTED_MEDIUM
1319,looking for internship,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.442,AUTO_ACCEPTED_MEDIUM
1412,New Grad | Looking for Full Time,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.565,AUTO_ACCEPTED_HIGH
1540,Project Manager & Operations / Unemployed,"Unemployed, Laid Off, Looking for Work, Seekin...",99-1001,99-0000,0.589,AUTO_ACCEPTED_HIGH


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Rename the title column
df_soc = soc_taxonomy.rename(columns={"Unnamed: 4": "title"}).copy()

# 2. Forward-fill the hierarchy so every row knows its Major Group
df_soc["Major Group"] = df_soc["Major Group"].ffill()
df_soc["Minor Group"] = df_soc["Minor Group"].ffill()

# 3. Filter to Detailed Occupations (or Broad Groups) - these are the actual job titles
# (e.g. "Software Developers", "Mechanical Engineers", "Accountants")
anchor_df = df_soc[
    df_soc["Detailed Occupation"].notna() | df_soc["Broad Group"].notna()
].copy()
anchor_df = anchor_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

# 4. Load the instruction model
model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-1.5B-instruct")

# 5. Embed the official titles from your table
print("Embedding official SOC titles...")
anchor_embeddings = model.encode(
    anchor_df["title"].tolist(), normalize_embeddings=True, show_progress_bar=True
)

# 6. Embed your candidate resume roles
print("Embedding your resume roles...")
role_embeddings = model.encode(
    all_unique_roles,
    prompt="Represent this career title for clustering by occupation and industry: ",
    normalize_embeddings=True,
    show_progress_bar=True,
)

# 7. Compute Cosine Similarity Matrix
sim_matrix = role_embeddings @ anchor_embeddings.T
best_match_indices = sim_matrix.argmax(axis=1)
confidence_scores = sim_matrix.max(axis=1)

# 8. Create the final classified DataFrame
matched_anchors = anchor_df.iloc[best_match_indices].reset_index(drop=True)

df_classified = pd.DataFrame(
    {
        "raw_role": all_unique_roles,
        "matched_soc_title": matched_anchors["title"],
        "detailed_soc_code": matched_anchors["Detailed Occupation"].combine_first(
            matched_anchors["Broad Group"]
        ),
        "major_group_code": matched_anchors["Major Group"],
        "confidence": confidence_scores.round(3),
    }
)

# Inspect results
print(df_classified.head(15))

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6155.29it/s]
[transformers] Qwen2Model LOAD REPORT from: Alibaba-NLP/gte-Qwen2-1.5B-instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding official SOC titles...


Batches: 100%|██████████| 34/34 [00:27<00:00,  1.24it/s]


Embedding your resume roles...


Batches: 100%|██████████| 244/244 [05:04<00:00,  1.25s/it]


                                     raw_role  \
0   Account Executive/Customer Success Hybrid   
1                                 Sub Teacher   
2                       Art Director/Laid Off   
3                       Art Director-Laid Off   
4                     Cyber Security Engineer   
5                                     Student   
6                                  Unemployed   
7            Enterprise Technology Consultant   
8                              College Senior   
9                        Customer Service Rep   
10                Administrative Assistant II   
11                         Admin Assistant II   
12                              Self-Employed   
13                     Restaurant/Bar manager   
14                            Retail Services   

                                    matched_soc_title detailed_soc_code  \
0   Switchboard Operators, Including Answering Ser...           43-2010   
1                Miscellaneous Postsecondary Teachers           2

In [ ]:
df_classified["major_group_code"].value_counts()

major_group_code
15-0000    1802
13-0000     976
25-0000     702
43-0000     675
17-0000     622
11-0000     614
41-0000     503
27-0000     319
19-0000     252
51-0000     221
53-0000     185
21-0000     164
29-0000     129
31-0000     123
49-0000     117
55-0000      88
39-0000      80
35-0000      59
47-0000      53
23-0000      50
33-0000      25
45-0000      16
37-0000      11
Name: count, dtype: int64

In [ ]:
df_classified[df_classified["major_group_code"] == "15-0000"]["matched_soc_title"].value_counts()

matched_soc_title
Software and Web Developers, Programmers, and Testers    1014
Miscellaneous Computer Occupations                        568
Network and Computer Systems Administrators                87
Computer Occupations, All Other                            45
Software Quality Assurance Analysts and Testers            41
Miscellaneous Mathematical Science Occupations             19
Database and Network Administrators and Architects         11
Mathematical Science Occupations, All Other                11
Data Scientists                                             3
Information Security Analysts                               2
Computer Support Specialists                                1
Name: count, dtype: int64

In [ ]:
df_classified[df_classified["detailed_soc_code"] == "15-1290"]["raw_role"].value_counts()

raw_role
Unemployed                             1
Self-Employed                          1
Graduate Student/Employed              1
Recent Unemployed                      1
Laid Off / Currently in Non-IT Work    1
                                      ..
ML/DS                                  1
business field                         1
Applied ML                             1
Business Field                         1
Data Science Co-op                     1
Name: count, Length: 568, dtype: int64

In [ ]:
df_classified[df_classified["major_group_code"] == "15-0000"]

,raw_role,matched_soc_title,detailed_soc_code,major_group_code,confidence
6,Unemployed,Miscellaneous Computer Occupations,15-1290,15-0000,0.547
12,Self-Employed,Miscellaneous Computer Occupations,15-1290,15-0000,0.580
19,Graduate Student/Employed,Miscellaneous Computer Occupations,15-1290,15-0000,0.588
20,Recent Unemployed,Miscellaneous Computer Occupations,15-1290,15-0000,0.570
22,Laid Off / Currently in Non-IT Work,Miscellaneous Computer Occupations,15-1290,15-0000,0.568
...,...,...,...,...,...
7773,business field,Miscellaneous Computer Occupations,15-1290,15-0000,0.516
7774,Applied ML,Miscellaneous Computer Occupations,15-1290,15-0000,0.429
7779,Business Field,Miscellaneous Computer Occupations,15-1290,15-0000,0.576
7780,Data Science Co-op,Miscellaneous Computer Occupations,15-1290,15-0000,0.430
